# NFL Player Prop Evaluation with SportsQuant

This notebook demonstrates how to use SportsQuant's NFL evaluation engine to analyze player props. We'll use **nflfastR** data (fetched automatically from GitHub releases) to build statistical models and compute expected value for NFL player props across PrizePicks, Underdog, and FanDuel.

**What you'll learn:**
- Fetching NFL player stats from nflfastR
- Building statistical models for player props
- Computing EV and Kelly fractions
- Comparing evaluations across DFS sites
- Understanding Poisson vs Normal distributions
- Correlation analysis for multi-leg entries

In [ ]:
import sys
from pathlib import Path

# Add sportsquant to path if running from notebooks directory
sys.path.insert(0, str(Path.cwd().parent.parent / "src"))

from sportsquant.models.analysis.evaluators.nfl_eval import (
    NFLEvaluator, NFLStatisticalModel, NFLDataProvider,
    NFLPrizePicksEvaluator, NFLUnderdogEvaluator, NFLFanDuelEvaluator,
)
from sportsquant.models.analysis.rules.nfl import (
    NFL_PASSING_STATS, NFL_RUSHING_STATS, NFL_RECEIVING_STATS,
    get_nfl_stat_key, is_nfl_poisson_stat, get_correlation_nfl,
    calculate_nfl_fanduel_points, FANDUEL_NFL_WEIGHTS,
)
import pandas as pd

print("✅ SportsQuant NFL modules loaded successfully")

In [ ]:
# Initialize the NFL data provider with disk caching
# nflfastR data is fetched from GitHub releases and cached locally
cache_dir = Path("./cache/nfl")

try:
    provider = NFLDataProvider(cache_dir=cache_dir, n_lookback=10)
    print(f"Cache directory: {cache_dir.absolute()}")
    print(f"Lookback window: {provider.n_lookback} games")
    print("✅ NFLDataProvider initialized")
except Exception as e:
    print(f"⚠️ Failed to initialize NFLDataProvider: {e}")
    print("nflfastR data may not be available offline. Fallback data will be used.")
    provider = None

In [ ]:
# Fetch recent game log for Patrick Mahomes
import numpy as np

DATA_AVAILABLE = False
gamelog = pd.DataFrame()

try:
    if provider is not None:
        player_name = "Patrick Mahomes"
        player_id = provider.get_player_id(player_name)

        if player_id:
            gamelog = provider.get_gamelog(player_id, lookback=10)
            if not gamelog.empty:
                DATA_AVAILABLE = True
                print(f"Player: {player_name}")
                print(f"Player ID: {player_id}")
                print(f"Games fetched: {len(gamelog)}")

                # Display key passing stats
                cols = ["season", "week", "passing_yards", "passing_tds", "interceptions",
                        "completions", "attempts"]
                available_cols = [c for c in cols if c in gamelog.columns]
                display(gamelog[available_cols].tail(10))
            else:
                print(f"⚠️ Empty gamelog for {player_name}")
        else:
            print(f"⚠️ Could not find player ID for {player_name}")
    else:
        print("⚠️ NFLDataProvider not available")
except Exception as e:
    print(f"⚠️ Error fetching data: {e}")

if not DATA_AVAILABLE:
    print("\n--- Using fallback mock data (nflfastR not available offline) ---")
    np.random.seed(42)
    gamelog = pd.DataFrame({
        "season": [2024] * 10,
        "week": list(range(1, 11)),
        "passing_yards": np.random.normal(280, 60, 10).round(0),
        "passing_tds": np.random.poisson(2.5, 10),
        "interceptions": np.random.poisson(0.5, 10),
        "rushing_yards": np.random.normal(20, 10, 10).round(0),
        "rushing_tds": np.random.poisson(0.2, 10),
        "receiving_yards": [0]*10,
        "receptions": [0]*10,
        "receiving_tds": [0]*10,
    })
    display(gamelog)

In [ ]:
# Build statistical models for Mahomes' passing stats
models = {}

try:
    model = NFLStatisticalModel(data_provider=provider, base_blend=0.30)
    stat_types = ["passing_yards", "passing_tds"]
    models = model.build_model("Patrick Mahomes", stat_types, position="QB")

    if models:
        for stat, model_dict in models.items():
            print(f"\n📊 {stat}:")
            for key, value in model_dict.items():
                if isinstance(value, float) and not np.isnan(value):
                    print(f"  {key}: {value:.2f}")
                else:
                    print(f"  {key}: {value}")
    else:
        raise RuntimeError("build_model returned empty dict")
except Exception as e:
    print(f"⚠️ Could not build model from live data: {e}")
    print("Building model from fallback mock data instead...")

    # Fallback: build model directly from mock gamelog
    model = NFLStatisticalModel(data_provider=provider, base_blend=0.30)
    models = model.build_models_from_gamelog(gamelog, ["passing_yards", "passing_tds"])

    for stat, model_dict in models.items():
        print(f"\n📊 {stat}:")
        for key, value in model_dict.items():
            if isinstance(value, float) and not (isinstance(value, float) and np.isnan(value)):
                print(f"  {key}: {value:.2f}")
            else:
                print(f"  {key}: {value}")

if not models:
    print("⚠️ No models available. The rest of the notebook will use defaults.")

In [ ]:
# Calculate probability of Mahomes going over common prop lines
lines = {
    "passing_yards": 280.5,
    "passing_tds": 2.5,
}

print("Probability Estimates:\n")
for stat, line in lines.items():
    if stat in models:
        try:
            prob_over = model.calc_prob_over(models[stat], line)
            if np.isnan(prob_over):
                print(f"{stat}: Over {line} → model returned NaN")
            else:
                prob_under = 1.0 - prob_over
                print(f"{stat}: Over {line} → {prob_over:.1%} | Under {line} → {prob_under:.1%}")
        except Exception as e:
            print(f"{stat}: Error calculating probability: {e}")
    else:
        print(f"{stat}: Model not available")

In [ ]:
# Evaluate a PrizePicks projection for Mahomes passing yards
from dataclasses import dataclass

@dataclass
class MockProjection:
    player_name: str
    stat_type: str
    line: float
    site: str
    odds_over: int = -110
    odds_under: int = -110

try:
    evaluator = NFLEvaluator(
        model_weight=0.30,
        min_confidence=45.0,
        min_edge=0.03,
        stat_model=model,
    )

    proj = MockProjection(
        player_name="Patrick Mahomes",
        stat_type="passing_yards",
        line=280.5,
        site="PrizePicks",
    )

    result = evaluator.evaluate_projection(proj, site="PrizePicks")

    print(f"📊 Evaluation Result:")
    print(f"  Player: {result.player_name}")
    print(f"  Stat: {result.stat_type}")
    print(f"  Line: {result.line}")
    print(f"  Model Probability: {result.model_prob:.1%}")
    print(f"  Fair Probability: {result.fair_prob:.1%}")
    print(f"  Final Probability: {result.final_prob:.1%}")
    print(f"  Edge: {result.edge:.1%}")
    print(f"  EV per $1: ${result.ev:.3f}")
    print(f"  Kelly Fraction: {result.kelly_fraction:.1%}")
    print(f"  Confidence: {result.confidence:.1f}%")
    print(f"  Recommended Side: {result.recommended_side}")
except Exception as e:
    print(f"⚠️ Error evaluating projection: {e}")
    print("This is expected if model data is not available.")

In [ ]:
# Compare evaluations across PrizePicks, Underdog, and FanDuel
try:
    pp_eval = NFLPrizePicksEvaluator(model_weight=0.30, stat_model=model)
    ud_eval = NFLUnderdogEvaluator(model_weight=0.30, stat_model=model)
    fd_eval = NFLFanDuelEvaluator(model_weight=0.30, stat_model=model)

    sites = {
        "PrizePicks": pp_eval,
        "Underdog": ud_eval,
        "FanDuel": fd_eval,
    }

    results = []
    for site_name, site_eval in sites.items():
        try:
            r = site_eval.evaluate_projection(proj, site=site_name)
            results.append({
                "Site": site_name,
                "EV": f"${r.ev:.3f}",
                "Edge": f"{r.edge:.1%}",
                "Kelly": f"{r.kelly_fraction:.1%}",
                "Side": r.recommended_side,
            })
        except Exception as e:
            results.append({
                "Site": site_name,
                "EV": "N/A",
                "Edge": "N/A",
                "Kelly": "N/A",
                "Side": str(e)[:50],
            })

    display(pd.DataFrame(results))
except Exception as e:
    print(f"⚠️ Error in per-site comparison: {e}")

In [ ]:
# Rank multiple mock projections by EV
projections = [
    MockProjection("Patrick Mahomes", "passing_yards", 280.5, "PrizePicks"),
    MockProjection("Patrick Mahomes", "passing_tds", 2.5, "PrizePicks"),
    MockProjection("Patrick Mahomes", "passing_yards", 260.5, "PrizePicks"),
    MockProjection("Patrick Mahomes", "passing_tds", 1.5, "PrizePicks"),
]

try:
    ranked = evaluator.rank_projections(projections, site="PrizePicks")

    if ranked:
        rank_data = []
        for i, result in enumerate(ranked, 1):
            rank_data.append({
                "Rank": i,
                "Player": result.player_name,
                "Stat": result.stat_type,
                "Line": result.line,
                "EV": f"${result.ev:.3f}",
                "Edge": f"{result.edge:.1%}",
                "Side": result.recommended_side,
            })
        display(pd.DataFrame(rank_data))
    else:
        print("No projections met evaluation criteria")
except Exception as e:
    print(f"⚠️ Error ranking projections: {e}")

In [ ]:
# Filter to only the best picks
try:
    best = evaluator.get_best_single_picks(
        projections, min_ev=0.05, min_confidence=50.0, site="PrizePicks"
    )

    if best:
        best_data = []
        for result in best:
            best_data.append({
                "Player": result.player_name,
                "Stat": result.stat_type,
                "Line": result.line,
                "EV": f"${result.ev:.3f}",
                "Confidence": f"{result.confidence:.1f}%",
                "Side": result.recommended_side,
            })
        display(pd.DataFrame(best_data))
    else:
        print("No picks met the minimum EV and confidence thresholds")
except Exception as e:
    print(f"⚠️ Error getting best picks: {e}")

### Distribution Types

SportsQuant automatically selects the appropriate distribution for each stat type:

- **Normal distribution**: Used for continuous stats (yards, completions, attempts)
- **Poisson distribution**: Used for count-based stats (TDs, INTs, sacks, tackles, fumbles)

This matters because Poisson distributions have different tail behavior — a player might have a 5% chance of throwing 4+ TDs even if their average is 2.0.

In [ ]:
# Check which stats use Poisson distribution
test_stats = [
    "passing_yards", "passing_tds", "interceptions",
    "rushing_yards", "rushing_tds", "receptions",
    "receiving_yards", "receiving_tds", "sacks",
]

print("Distribution Type by Stat:\n")
for stat in test_stats:
    stat_key = get_nfl_stat_key(stat)
    is_poisson = is_nfl_poisson_stat(stat_key)
    dist_type = "Poisson" if is_poisson else "Normal"
    print(f"  {stat_key:25s} → {dist_type}")

In [ ]:
# Analyze correlations between common stat pairs
correlation_pairs = [
    ("passing_yards", "passing_tds", True, True),   # Same player, same game
    ("passing_yards", "rushing_yards", True, True),  # Same player, same game
    ("receiving_yards", "receiving_tds", True, True),
    ("passing_yards", "receiving_yards", False, True),  # Different players, same game
]

print("Correlation Analysis:\n")
for stat1, stat2, same_player, same_game in correlation_pairs:
    corr = get_correlation_nfl(stat1, stat2, same_player=same_player, same_game=same_game)
    player_label = "Same Player" if same_player else "Different Players"
    game_label = "Same Game" if same_game else "Different Games"
    print(f"  {stat1} ↔ {stat2} ({player_label}, {game_label}): r = {corr:.3f}")

In [ ]:
# Calculate FanDuel fantasy points from a stat line
# FANDUEL_NFL_WEIGHTS uses Title Case keys (e.g., "Passing Yards").
# We use direct weight lookup to compute fantasy points.
example_stats = {
    "Passing Yards": 300,
    "Passing TDs": 3,
    "Interceptions": 1,
    "Rushing Yards": 25,
    "Rushing TDs": 0,
}

fd_points = 0.0
for stat, value in example_stats.items():
    weight = FANDUEL_NFL_WEIGHTS.get(stat, 0.0)
    fd_points += value * weight

print("FanDuel Fantasy Points Calculation:\n")
for stat, value in example_stats.items():
    weight = FANDUEL_NFL_WEIGHTS.get(stat, 0.0)
    if weight != 0.0:
        print(f"  {stat}: {value} x {weight} = {value * weight:.1f}")
    else:
        print(f"  {stat}: {value} (no weight)")

print(f"\n  Total FanDuel Points: {fd_points:.1f}")

# Show the scoring weights
print("\nFanDuel NFL Scoring Weights (non-zero):")
for stat, weight in sorted(FANDUEL_NFL_WEIGHTS.items()):
    if weight != 0.0:
        print(f"  {stat}: {weight}")

## Summary

In this notebook, we:
1. ✅ Initialized the NFL data provider with nflfastR data
2. ✅ Fetched player game logs and built statistical models
3. ✅ Calculated probabilities for common prop lines
4. ✅ Evaluated projections across PrizePicks, Underdog, and FanDuel
5. ✅ Ranked projections by expected value
6. ✅ Analyzed stat correlations for multi-leg entries
7. ✅ Computed FanDuel fantasy points

## Next Steps

- **02_nfl_backtest.ipynb** — Backtest NFL betting strategies with historical data
- **03_nfl_ratings.ipynb** — Compute NFL team power ratings
- **CLI**: Try `sportsquant nfl ev --player "Patrick Mahomes" --stat passing_yards --line 280.5`